In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from lib.data.datasets import AccRawDataset
from lib.data.dataloading import load_raw, load_nursing_5_class
from lib.config import RAW_DIR
from lib.modules import optimization_loop_xonly,sample_regnet, optimization_loop_multi_class
from lib.models import RegNetv3, RegNetMAEv3, CosineMSELoss, RegNetv3Ci
from pathlib import Path
import json

In [70]:
CONFIG = {
    'WINDOW_SIZE':2001,
    'WINDOW_STRIDE':2001 // 16,
    'NURSING_STRIDE': 2001 // 16,
    'BATCH_SIZE': 512,
    'LEARNING_RATE': 1e-3,
    'CLASS_LR': 3e-4,
    'ENC_LEARNING_RATE': 5e-5,
    'TEST_SIZE': 0.1,
    'NURSING_TEST_SIZE': 0.25,
    'DEVICE': 'cuda:1',
    'DEPTHI': [2],
    'WIDTHI': [64],
    'NTL': 1,
    'DMODEL': 0,
    'MASKPCT': 0.5,
    'PDROPOUT': 0.0,
    'FREEZE': False,
    'WEIGHTS_FILE': None,
    'LSTM_SEQLEN': 3,
}
CONFIG['PRETRAINED'] = bool(CONFIG['WEIGHTS_FILE'])

In [16]:
class ClassifierLSTM(nn.Module):
    def __init__(self, CONFIG):
        super().__init__()
        self.weights_file = CONFIG.get('CLASS_WEIGHTS_FILE', None)
        if not self.weights_file:
            raise ValueError('No weights file provided')
        self.winsize = CONFIG['WINDOW_SIZE']
        self.seq_len = CONFIG['LSTM_SEQLEN']
        hidden_dim = CONFIG.get('HIDDEN_DIM', 64)
        dropout = CONFIG.get('LSTM_DROP', 0.0)

        # self.regnet = RegNetv3(CONFIG=CONFIG)
        self.regnet = RegNetv3Ci(CONFIG=CONFIG)
        print(f'Loading weights from {self.weights_file}')
        self.regnet.load_state_dict(torch.load(self.weights_file))
        for p in self.regnet.parameters():
            p.requires_grad = False
        
        self.lstm = nn.LSTM(
            input_size=5,   # number of classes
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.out = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim*2, 5)
        )
        
    def forward(self, x):
        x = x.view(x.shape[0], 3, self.seq_len, self.winsize).permute(0,2,1,3)
        ys = []
        for i in range(self.seq_len):
            ys.append(self.regnet(x[:,i]).unsqueeze(1))
        ys = torch.cat(ys,dim=1)
        o, (h,c) = self.lstm(ys)
        # get middle hidden state
        x = self.out(o[:,self.seq_len//2])
        return x

In [18]:
# model_dir = Path('/home/musa/eating-detection/dev/9_regnet-mae/prototyping-3-16-24/class2/class-pretrained')
model_dir = Path('/home/musa/eating-detection/dev/9_regnet-mae/random-search-class-fixed/[4, 13, 3]-[48, 120, 304]-pretrained-ci')
CONFIG = json.load(open(model_dir / 'config.json'))
CONFIG['LSTM_SEQLEN'] = 7
CONFIG['CLASS_WEIGHTS_FILE'] = str(model_dir / 'best_model.pt')
CONFIG['LSTM_HIDDEN'] = 8
CONFIG['LSTM_DROP'] = 0.25
CONFIG['BATCH_SIZE'] = 512
CONFIG['NURSING_STRIDE'] = 2001

nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE']*CONFIG['LSTM_SEQLEN'], 
    test_size=CONFIG['NURSING_TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['NURSING_STRIDE'],
)

model = ClassifierLSTM(CONFIG).to(CONFIG['DEVICE'])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['CLASS_LR'])


# for X, y in nursing_trainloader:
#     X, y = X.to(CONFIG['DEVICE']), y.to(CONFIG['DEVICE'])
#     print(X.shape)
#     yhat = model(X)
#     print(yhat.shape)
#     break
lstm_outdir = Path('/home/musa/eating-detection/dev/9_regnet-mae/random-search-lstm/[4, 13, 3]-[48, 120, 304]-poster-stride' + str(CONFIG['NURSING_STRIDE']))
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=500,
    device=CONFIG['DEVICE'],
    patience=50,
    outdir=lstm_outdir,
    writer=lstm_outdir,
    config=CONFIG
)

latent dim: 125
Model is loading pretrained encoder
Loading weights from /home/musa/eating-detection/dev/9_regnet-mae/random-search-class-fixed/[4, 13, 3]-[48, 120, 304]-pretrained-ci/best_model.pt


: Epoch 189: Train Loss: 0.32183: Dev Loss: 0.70441, Dev F1: 0.79727:  38%|███▊      | 190/500 [12:13<19:56,  3.86s/it]


KeyboardInterrupt: 